## Step 1 — Verify Java version (must say 11.x.x)

In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.31.11-hotspot"

In [2]:
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)
# Must say: openjdk version "11.x.x"
# If it still says 25, go fix JAVA_HOME in System Environment Variables and restart Jupyter

openjdk version "25.0.3" 2026-04-21 LTS
OpenJDK Runtime Environment Temurin-25.0.3+9 (build 25.0.3+9-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.3+9 (build 25.0.3+9-LTS, mixed mode, sharing)



## Step 2 — Verify PySpark version (must be 3.x.x, not 4.x.x)

In [3]:
import sys

# Uninstall PySpark 4 and install PySpark 3.5 (stable with Java 11)
# Run this cell ONCE, then comment it out
!{sys.executable} -m pip uninstall pyspark -y
!{sys.executable} -m pip install pyspark==3.5.1 --force-reinstall


# Must say: 3.5.1

  Using cached pyspark-3.5.1-py2.py3-none-any.whl
  Using cached py4j-0.10.9.7-py2.py3-none-any.whl.metadata (1.5 kB)
Using cached py4j-0.10.9.7-py2.py3-none-any.whl (200 kB)

  Attempting uninstall: py4j

    Found existing installation: py4j 0.10.9.7

    Uninstalling py4j-0.10.9.7:

      Successfully uninstalled py4j-0.10.9.7

   ---------------------------------------- 0/2 [py4j]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- ------------------- 1/2 [pyspark]
   -------------------- --------

In [4]:
# Verify versions first
import subprocess, pyspark

java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
print("Java:", java.split("\n")[0])
print("PySpark:", pyspark.__version__)

# Java must say 11.x.x
# PySpark must say 3.5.1

Java: openjdk version "25.0.3" 2026-04-21 LTS
PySpark: 3.5.1


## Step 3 — Start Spark Session

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LiverDiseaseAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark started successfully!")

Spark version: 3.5.1
Spark started successfully!


## Step 4 — Load the CSV dataset

In [6]:
import os

# Make sure CSV is in the same folder as this notebook
# Or provide the full path like: r"C:\Users\mooda\Downloads\Training_Liver_Disease_Dataset.csv"
csv_path = "./Training_Liver_Disease_Dataset.csv"

# Check if file exists
if not os.path.exists(csv_path):
    print(f"File not found: {csv_path}")
    print("Current directory:", os.getcwd())
    print("Files here:", os.listdir("."))
else:
    print("File found! Loading...")
    df = spark.read.csv(csv_path, header=True, inferSchema=True)
    print("Loaded successfully! Shape:", (df.count(), len(df.columns)))

File found! Loading...
Loaded successfully! Shape: (30000, 33)


## Step 5 — Basic Exploration

In [7]:
# Schema
df.printSchema()

root
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Obesity_Class: string (nullable = true)
 |-- Waist_Circumference: double (nullable = true)
 |-- Diet_Quality: string (nullable = true)
 |-- Physical_Activity: string (nullable = true)
 |-- Sleep_Hours: double (nullable = true)
 |-- Smoking_Status: string (nullable = true)
 |-- Alcohol_Consumption: string (nullable = true)
 |-- Sym_Fatigue: integer (nullable = true)
 |-- Sym_Jaundice: integer (nullable = true)
 |-- Sym_Abdominal_Pain: integer (nullable = true)
 |-- Sym_Itching: integer (nullable = true)
 |-- Sym_Ascites: integer (nullable = true)
 |-- Sym_Dark_Urine: integer (nullable = true)
 |-- Sym_Weight_Loss: integer (nullable = true)
 |-- Comorb_Diabetes: integer (nullable = true)
 |-- Comorb_Hypertension: integer (nullable = true)
 |-- Comorb_Genetic_History: integer (nullable = true)
 |-- ALT: double (nullable = true)
 |

In [8]:
# First 5 rows
df.show(5, truncate=False)

+---+------+-----------------+----+-------------+-------------------+------------+-----------------+-----------+--------------+-------------------+-----------+------------+------------------+-----------+-----------+--------------+---------------+---------------+-------------------+----------------------+-----+-----+---------+-------+---------+---------------+---+-------------+----+-------------------+----------------------+---------------------------+
|Age|Gender|Occupation       |BMI |Obesity_Class|Waist_Circumference|Diet_Quality|Physical_Activity|Sleep_Hours|Smoking_Status|Alcohol_Consumption|Sym_Fatigue|Sym_Jaundice|Sym_Abdominal_Pain|Sym_Itching|Sym_Ascites|Sym_Dark_Urine|Sym_Weight_Loss|Comorb_Diabetes|Comorb_Hypertension|Comorb_Genetic_History|ALT  |AST  |Bilirubin|Albumin|Platelets|Alk_Phosphatase|GGT|Triglycerides|INR |Medication_History |Source                |Liver_Disease_Class        |
+---+------+-----------------+----+-------------+-------------------+------------+------

In [9]:
# Basic statistics
df.describe().show()

+-------+------------------+------+----------+------------------+-------------+-------------------+------------+-----------------+------------------+--------------+-------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+-------------------+----------------------+------------------+------------------+------------------+-------------------+------------------+-----------------+-----------------+----------------+------------------+-------------------+------+--------------------+
|summary|               Age|Gender|Occupation|               BMI|Obesity_Class|Waist_Circumference|Diet_Quality|Physical_Activity|       Sleep_Hours|Smoking_Status|Alcohol_Consumption|       Sym_Fatigue|      Sym_Jaundice| Sym_Abdominal_Pain|        Sym_Itching|        Sym_Ascites|     Sym_Dark_Urine|    Sym_Weight_Loss|   Comorb_Diabetes|Comorb_Hypertension|Comorb_Genetic_History|               

In [10]:
# Check null values per column
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()

+---+------+----------+---+-------------+-------------------+------------+-----------------+-----------+--------------+-------------------+-----------+------------+------------------+-----------+-----------+--------------+---------------+---------------+-------------------+----------------------+---+---+---------+-------+---------+---------------+---+-------------+---+------------------+------+-------------------+
|Age|Gender|Occupation|BMI|Obesity_Class|Waist_Circumference|Diet_Quality|Physical_Activity|Sleep_Hours|Smoking_Status|Alcohol_Consumption|Sym_Fatigue|Sym_Jaundice|Sym_Abdominal_Pain|Sym_Itching|Sym_Ascites|Sym_Dark_Urine|Sym_Weight_Loss|Comorb_Diabetes|Comorb_Hypertension|Comorb_Genetic_History|ALT|AST|Bilirubin|Albumin|Platelets|Alk_Phosphatase|GGT|Triglycerides|INR|Medication_History|Source|Liver_Disease_Class|
+---+------+----------+---+-------------+-------------------+------------+-----------------+-----------+--------------+-------------------+-----------+------------+

In [11]:
# Stop Spark when done
spark.stop()
print("Spark session stopped.")

Spark session stopped.
